# Independent check

Recomputes, without importing `pipeline/`, the Gini and KS of the committed test sample and the points of 5 borrowers straight from `public/data/scorecard.json`. Compare the printed values with `metrics.json` and `test_sample.csv`.

In [ ]:
import json, math
import numpy as np
import pandas as pd

card = json.load(open('../public/data/scorecard.json'))
sample = pd.read_csv('../data/sample/test_sample.csv')
len(sample), sample['bad'].mean()

## Gini and KS by hand

AUC = share of (bad, good) pairs where the bad has the lower score; ties count half. Gini = 2 × AUC − 1. KS = largest gap between the cumulative score distributions of bads and goods.

In [ ]:
s = sample['score_python'].to_numpy(); y = sample['bad'].to_numpy()
bad, good = s[y == 1], s[y == 0]
# pairwise comparison (3 000 loans → about 1 million pairs, fine)
lower = (bad[:, None] < good[None, :]).sum() + 0.5 * (bad[:, None] == good[None, :]).sum()
auc = lower / (len(bad) * len(good))
thr = np.sort(np.unique(s))
cdf_b = np.array([(bad <= t).mean() for t in thr]); cdf_g = np.array([(good <= t).mean() for t in thr])
ks = np.max(np.abs(cdf_b - cdf_g))
print(f'AUC {auc:.4f}  Gini {2*auc-1:.4f}  KS {ks:.4f}')

## Points for 5 borrowers, read off the scorecard table

In [ ]:
def find_bin(v, x):
    if v['type'] == 'numeric':
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return len(v['bins']) - 1
        k = 0
        while k < len(v['cuts']) and x >= v['cuts'][k]:
            k += 1
        return k
    x = 'Missing' if x is None or (isinstance(x, float) and math.isnan(x)) else str(x)
    for i, b in enumerate(v['bins']):
        if x in b['categories']:
            return i
    return next(i for i, b in enumerate(v['bins']) if '__other__' in b['categories'])

rows = []
for _, r in sample.head(5).iterrows():
    pts = {v['name']: v['bins'][find_bin(v, r[v['name']])]['points'] for v in card['variables']}
    total = card['base_points'] + sum(pts.values())
    odds = math.exp((total - card['scaling']['offset']) / card['scaling']['factor'])
    rows.append({'id': r['id'], **{k: round(p, 2) for k, p in pts.items()}, 'score': round(total, 4),
                 'pipeline': round(r['score_python'], 4), 'PD': round(max(1 / (1 + odds), card['pd_floor']), 4)})
pd.DataFrame(rows)